# 04 - Forecasting

Trains LightGBM forecasting models on 2023 data and produces 366 daily predictions
per household for 2024.

**Three configurations:**
| Config | Description |
|---|---|
| `feature_ward` | One model per Feature+Ward cluster (cluster-level) |
| `kshape` | One model per k-Shape cluster (cluster-level) |
| `global` | Single model on all households, no clustering (dataset-level baseline) |

**Data leakage note:**
- `household_means` (used as a feature) are computed from 2023 train data only.
- Lag features for Jan 2024 look back into Dec 2023 - this uses 2023 *consumption* as context, not 2024 targets. Safe.
- Model parameter selection is performed using rolling validation on 2023 only; 2024 is used only as final ground truth evaluation.
- 2024 values are never seen during model fitting.

Output prediction CSVs are saved to results/predictions/.

Evaluation outputs and 2023-only backtest selection tables are saved to results/evaluation/.

## Setup

In [1]:
import sys, os, warnings
warnings.filterwarnings("ignore")

project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd
from pathlib import Path

from src.utils.config import (
    RESULTS_DIR,
    PREDICTIONS_DIR,
)
from src.utils.data_loader import load_train_data, load_test_data
from src.forecasting.pipeline import (
    CLUSTERING_CONFIG,
    run_cluster_pipeline,
    run_global_pipeline,
)

PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / "evaluation").mkdir(parents=True, exist_ok=True)

print("Environment ready.")
print("Selected clustering config:", CLUSTERING_CONFIG)

Environment ready.
Selected clustering config: {'feature_ward': {'k': 3}, 'kshape': {'k': 2}}


## Load Data

In [3]:
train_wide = load_train_data()   # 2023 - used for training only
test_wide  = load_test_data()    # 2024 - used for inference only

print(f'Train (2023): {train_wide.shape}  - households × days')
print(f'Test  (2024): {test_wide.shape}  - households × days')
assert set(train_wide.index) == set(test_wide.index), 'Household ID mismatch!'
print(f'Households: {len(train_wide):,}  ✓')

Train (2023): (17547, 365)  - households × days
Test  (2024): (17547, 366)  - households × days
Households: 17,547  ✓


## Run forecasting pipelines

This notebook now uses the updated forecasting modules directly instead of
reimplementing the forecasting flow inside notebook cells.

The updated pipeline now handles:

- loading forecasting-ready nonnegative cluster handoff labels
- 2023-only feature construction
- cluster-derived seasonal prior features
- 2023-only walk-forward backtest parameter selection
- final training on full 2023
- recursive 2024 forecasting
- assignment-compliant 2024 household-level MAE evaluation

In [5]:
print("Selected k values from Task 1:")
print(f"  feature_ward: k = {CLUSTERING_CONFIG['feature_ward']['k']}")
print(f"  kshape      : k = {CLUSTERING_CONFIG['kshape']['k']}")

Selected k values from Task 1:
  feature_ward: k = 3
  kshape      : k = 2


## Feature+Ward forecasting

In [4]:
%%time
method_fw = "feature_ward"
k_fw = CLUSTERING_CONFIG[method_fw]["k"]

preds_fw = run_cluster_pipeline(
    method=method_fw,
    k=k_fw,
    train_wide=train_wide,
    test_wide=test_wide,
    save_models=True,
)

out_fw = PREDICTIONS_DIR / f"predictions_2024_{method_fw}.csv"
fw_eval_path = RESULTS_DIR / "evaluation" / f"evaluation_summary_{method_fw}.csv"
fw_hh_mae_path = RESULTS_DIR / "evaluation" / f"household_mae_{method_fw}.csv"
fw_backtest_path = RESULTS_DIR / "evaluation" / f"backtest_selection_{method_fw}.csv"

print("\nFeature+Ward outputs")
print(f"  Predictions : {out_fw}")
print(f"  Eval summary: {fw_eval_path}")
print(f"  HH MAE      : {fw_hh_mae_path}")
print(f"  Backtest    : {fw_backtest_path}")

display(preds_fw.head())
display(pd.read_csv(fw_eval_path))


  feature_ward  (k=3)
  [feature_ward k=3] 17,547 forecasting-ready households loaded

[1/4] Building 2023 training features...
  Train: (5878245, 28)
  Features (24): ['lag_1', 'lag_7', 'lag_14', 'lag_30', 'rolling_mean_7', 'rolling_std_7', 'rolling_mean_14', 'rolling_std_14', 'rolling_mean_30', 'rolling_std_30', 'day_of_week', 'month', 'day_of_year', 'is_weekend', 'week_of_year', 'day_of_year_sin', 'day_of_year_cos', 'week_of_year_sin', 'week_of_year_cos', 'household_mean', 'cluster_dow_mean', 'cluster_month_mean', 'household_minus_cluster_dow_mean', 'household_minus_cluster_month_mean']

[2/4] Selecting model params with 2023-only walk-forward validation...

  Cluster 0 backtest tuning
  [backtest] selected params: {'n_estimators': 300, 'learning_rate': 0.05, 'num_leaves': 31, 'min_child_samples': 20} | mean 2023 validation MAE = 3.867287

  Cluster 1 backtest tuning
  [backtest] selected params: {'n_estimators': 500, 'learning_rate': 0.03, 'num_leaves': 127, 'min_child_samples': 4

,household_id,date,predicted,cluster,model_used
0,22,2024-01-01,10.375867,1,cluster_lgbm
1,22,2024-01-02,10.144495,1,cluster_lgbm
2,22,2024-01-03,9.952144,1,cluster_lgbm
3,22,2024-01-04,10.232076,1,cluster_lgbm
4,22,2024-01-05,9.518654,1,cluster_lgbm


,method,n_households_evaluated,expected_days_per_household,n_households_with_complete_366_days,average_household_mae,median_household_mae
0,feature_ward,17547,366,17547,3.940402,2.230138


CPU times: total: 9h 47min
Wall time: 1h 1min 32s


## k-Shape forecasting

In [6]:
%%time
method_ks = "kshape"
k_ks = CLUSTERING_CONFIG[method_ks]["k"]

preds_ks = run_cluster_pipeline(
    method=method_ks,
    k=k_ks,
    train_wide=train_wide,
    test_wide=test_wide,
    save_models=True,
)

out_ks = PREDICTIONS_DIR / f"predictions_2024_{method_ks}.csv"
ks_eval_path = RESULTS_DIR / "evaluation" / f"evaluation_summary_{method_ks}.csv"
ks_hh_mae_path = RESULTS_DIR / "evaluation" / f"household_mae_{method_ks}.csv"
ks_backtest_path = RESULTS_DIR / "evaluation" / f"backtest_selection_{method_ks}.csv"

print("\nk-Shape outputs")
print(f"  Predictions : {out_ks}")
print(f"  Eval summary: {ks_eval_path}")
print(f"  HH MAE      : {ks_hh_mae_path}")
print(f"  Backtest    : {ks_backtest_path}")

display(preds_ks.head())
display(pd.read_csv(ks_eval_path))


  kshape  (k=2)
  [kshape k=2] 17,547 forecasting-ready households loaded

[1/4] Building 2023 training features...
  Train: (5878245, 28)
  Features (24): ['lag_1', 'lag_7', 'lag_14', 'lag_30', 'rolling_mean_7', 'rolling_std_7', 'rolling_mean_14', 'rolling_std_14', 'rolling_mean_30', 'rolling_std_30', 'day_of_week', 'month', 'day_of_year', 'is_weekend', 'week_of_year', 'day_of_year_sin', 'day_of_year_cos', 'week_of_year_sin', 'week_of_year_cos', 'household_mean', 'cluster_dow_mean', 'cluster_month_mean', 'household_minus_cluster_dow_mean', 'household_minus_cluster_month_mean']

[2/4] Selecting model params with 2023-only walk-forward validation...

  Cluster 0 backtest tuning
  [backtest] selected params: {'n_estimators': 300, 'learning_rate': 0.05, 'num_leaves': 31, 'min_child_samples': 20} | mean 2023 validation MAE = 2.801351

  Cluster 1 backtest tuning
  [backtest] selected params: {'n_estimators': 500, 'learning_rate': 0.03, 'num_leaves': 127, 'min_child_samples': 40} | mean 20

,household_id,date,predicted,cluster,model_used
0,22,2024-01-01,10.575288,0,cluster_lgbm
1,22,2024-01-02,10.122440,0,cluster_lgbm
2,22,2024-01-03,10.012242,0,cluster_lgbm
3,22,2024-01-04,10.256127,0,cluster_lgbm
4,22,2024-01-05,9.868169,0,cluster_lgbm


,method,n_households_evaluated,expected_days_per_household,n_households_with_complete_366_days,average_household_mae,median_household_mae
0,kshape,17547,366,17547,4.167071,2.273118


CPU times: total: 10h 44min 20s
Wall time: 1h 6min 28s


## Global baseline forecasting

This baseline uses no clustering structure for segmentation.

In [7]:
%%time
preds_gl = run_global_pipeline(
    train_wide=train_wide,
    test_wide=test_wide,
    save_models=True,
)

out_gl = PREDICTIONS_DIR / "predictions_2024_global.csv"
gl_eval_path = RESULTS_DIR / "evaluation" / "evaluation_summary_global.csv"
gl_hh_mae_path = RESULTS_DIR / "evaluation" / "household_mae_global.csv"
gl_backtest_path = RESULTS_DIR / "evaluation" / "backtest_selection_global.csv"

print("\nGlobal outputs")
print(f"  Predictions : {out_gl}")
print(f"  Eval summary: {gl_eval_path}")
print(f"  HH MAE      : {gl_hh_mae_path}")
print(f"  Backtest    : {gl_backtest_path}")

display(preds_gl.head())
display(pd.read_csv(gl_eval_path))


  global baseline (no clustering)

[1/4] Building 2023 training features...
  Train: (5878245, 28)

[2/4] Selecting global params with 2023-only walk-forward validation...
  [backtest] selected params: {'n_estimators': 500, 'learning_rate': 0.03, 'num_leaves': 127, 'min_child_samples': 40} | mean 2023 validation MAE = 2.448110
  Saved 2023 backtest selection table: c:\Users\cescedes\Documents\GitHub\rdkd\results\evaluation\backtest_selection_global.csv

[3/4] Training global model on full 2023 data...
  [point] global model: 5,878,245 train rows | params={'n_estimators': 500, 'learning_rate': 0.03, 'num_leaves': 127, 'min_child_samples': 40}
  Models saved to c:\Users\cescedes\Documents\GitHub\rdkd\results\models\global

[4/4] Predicting 2024...

  Saved predictions: c:\Users\cescedes\Documents\GitHub\rdkd\results\predictions\predictions_2024_global.csv
  Households: 17,547  |  Days per household: 366–366

[4/4] Assignment-compliant evaluation on 2024 ground truth
  Households evaluat

,household_id,date,predicted,cluster,model_used
0,22,2024-01-01,10.327396,0,global_lgbm
1,22,2024-01-02,10.180171,0,global_lgbm
2,22,2024-01-03,10.048242,0,global_lgbm
3,22,2024-01-04,10.368309,0,global_lgbm
4,22,2024-01-05,9.691458,0,global_lgbm


,method,n_households_evaluated,expected_days_per_household,n_households_with_complete_366_days,average_household_mae,median_household_mae
0,global,17547,366,17547,4.348562,2.322294


CPU times: total: 5h 42min 11s
Wall time: 42min 41s


## Output Summary

In [8]:
summary_paths = {
    "feature_ward": RESULTS_DIR / "evaluation" / "evaluation_summary_feature_ward.csv",
    "kshape": RESULTS_DIR / "evaluation" / "evaluation_summary_kshape.csv",
    "global": RESULTS_DIR / "evaluation" / "evaluation_summary_global.csv",
}

summary_tables = []
for method, path in summary_paths.items():
    df = pd.read_csv(path)
    df["method"] = method
    summary_tables.append(df)

comparison = pd.concat(summary_tables, ignore_index=True)
comparison = comparison[
    [
        "method",
        "n_households_evaluated",
        "expected_days_per_household",
        "n_households_with_complete_366_days",
        "average_household_mae",
        "median_household_mae",
    ]
].sort_values("average_household_mae", ascending=True).reset_index(drop=True)

print("=== FORECASTING COMPLETE ===")
print()
print("Prediction files:")
print(f"  {PREDICTIONS_DIR / 'predictions_2024_feature_ward.csv'}")
print(f"  {PREDICTIONS_DIR / 'predictions_2024_kshape.csv'}")
print(f"  {PREDICTIONS_DIR / 'predictions_2024_global.csv'}")
print()
print("Evaluation files:")
print(f"  {RESULTS_DIR / 'evaluation' / 'evaluation_summary_feature_ward.csv'}")
print(f"  {RESULTS_DIR / 'evaluation' / 'evaluation_summary_kshape.csv'}")
print(f"  {RESULTS_DIR / 'evaluation' / 'evaluation_summary_global.csv'}")
print()
print("Household-level MAE files:")
print(f"  {RESULTS_DIR / 'evaluation' / 'household_mae_feature_ward.csv'}")
print(f"  {RESULTS_DIR / 'evaluation' / 'household_mae_kshape.csv'}")
print(f"  {RESULTS_DIR / 'evaluation' / 'household_mae_global.csv'}")
print()
print("Backtest selection files:")
print(f"  {RESULTS_DIR / 'evaluation' / 'backtest_selection_feature_ward.csv'}")
print(f"  {RESULTS_DIR / 'evaluation' / 'backtest_selection_kshape.csv'}")
print(f"  {RESULTS_DIR / 'evaluation' / 'backtest_selection_global.csv'}")
print()

display(comparison)

=== FORECASTING COMPLETE ===

Prediction files:
  c:\Users\cescedes\Documents\GitHub\rdkd\results\predictions\predictions_2024_feature_ward.csv
  c:\Users\cescedes\Documents\GitHub\rdkd\results\predictions\predictions_2024_kshape.csv
  c:\Users\cescedes\Documents\GitHub\rdkd\results\predictions\predictions_2024_global.csv

Evaluation files:
  c:\Users\cescedes\Documents\GitHub\rdkd\results\evaluation\evaluation_summary_feature_ward.csv
  c:\Users\cescedes\Documents\GitHub\rdkd\results\evaluation\evaluation_summary_kshape.csv
  c:\Users\cescedes\Documents\GitHub\rdkd\results\evaluation\evaluation_summary_global.csv

Household-level MAE files:
  c:\Users\cescedes\Documents\GitHub\rdkd\results\evaluation\household_mae_feature_ward.csv
  c:\Users\cescedes\Documents\GitHub\rdkd\results\evaluation\household_mae_kshape.csv
  c:\Users\cescedes\Documents\GitHub\rdkd\results\evaluation\household_mae_global.csv

Backtest selection files:
  c:\Users\cescedes\Documents\GitHub\rdkd\results\evaluatio

,method,n_households_evaluated,expected_days_per_household,n_households_with_complete_366_days,average_household_mae,median_household_mae
0,feature_ward,17547,366,17547,3.940402,2.230138
1,kshape,17547,366,17547,4.167071,2.273118
2,global,17547,366,17547,4.348562,2.322294
